In [1]:
import numpy as np
import pandas as pd
import os
import allensdk.core.swc as swc
# from morph_utils.templates import load_layer_template
from lims_utils import get_swc_from_lims
from query import query_lims_for_layers
from morph import to_dict, dict_to_Morphology
from fiducials import get_coords, convert_coords_str, upright_angle
from geometry import line, intersection, find_translation, find_farthest, determine_mirror


In [2]:
csv =r'\\allen\programs\celltypes\workgroups\mousecelltypes\SarahWB\github_projects\primate_uprighting\testing\specimens.csv'
#raw/orginal file 
o_out = r'\\allen\programs\celltypes\workgroups\mousecelltypes\SarahWB\github_projects\primate_uprighting\testing\original'
upright_out = r'\\allen\programs\celltypes\workgroups\mousecelltypes\SarahWB\github_projects\primate_uprighting\testing\upright'

#make dirs if they don't exist 
if not os.path.isdir(upright_out): os.mkdir(upright_out)  
if not os.path.isdir(o_out): os.mkdir(o_out)

specimen_id_column = 'specimen_id'


In [3]:
# ##Check any conditions you need to HERE
df = pd.read_csv(csv)
df[specimen_id_column] = [a.replace(u'\u200b', '') for a in df[specimen_id_column].values]
print(len(df))
df.head()

20


,specimen_name,specimen_id,area,medial_side_to_viewerpov
0,Q21.26.013.12.03.01,1102613885,Pu,right
1,Q21.26.013.12.02.03​,1102623321,Pu,right
2,Q21.26.019.13.01.02,1110880399,Ca,right
3,Q21.26.019.13.01.05,1110896667,Ca,right
4,Q21.26.020.14.05.02,1112128140,ACB,left


In [4]:
#upright one cell
def upright_nrn(specimen_id, oout, uout, error_dict={}):

    try:

        print(specimen_id)

        ldf = query_lims_for_layers(specimen_id)

        try: soma_coords, pia_coords, wm_coords, layer_coords = get_coords(ldf, ["1", "2/3",  "4", "5", "6a", "6b"])
        except:
            print("ERROR: Couldn't load layers for {}".format(specimen_id))
            error_dict[specimen_id] = "Could not load layers"
            return error_dict

        try: swc_filename, swc_path = get_swc_from_lims(specimen_id)
        except TypeError:
            print("ERROR: Could not get swc from lims for ", specimen_id)
            error_dict[specimen_id] = "Could not get swc from lims"
            return error_dict
        
        swc_path = swc_path.replace('\\', '/')
        swc_path = swc_path.replace('/', '//', 1)
        morph = swc.read_swc(swc_path)
        morph.write(oout)

        if soma_coords is None:
            print("ERROR: No soma drawing for", specimen_id)
            error_dict[specimen_id] = "No soma drawing"
            return error_dict
        if pia_coords is None:
            print("ERROR: No 'pia' drawing for", specimen_id)
            error_dict[specimen_id] = "No 'pia' drawing"
            return error_dict
        if wm_coords is None:
            print("ERROR: No 'white matter' drawing for", specimen_id)
            error_dict[specimen_id] = "No 'white matter' drawing"
            return error_dict

        #Edit 'Pia' coords
        row = ldf[ldf.draw_type == 'Pia']
        res = row.res.values[0]  
        pcoords = row.poly_coords.values[0]
        plx, ply = convert_coords_str(pcoords)
        plx = plx * res
        ply = ply * res
        L1 = line([plx[0],ply[0]], [plx[-1], ply[-1]])

        #Add 'White Matter' coords
        row = ldf[ldf.draw_type == 'White Matter']
        res = row.res.values[0]  
        wcoords = row.poly_coords.values[0]
        lx, ly = convert_coords_str(wcoords)
        lx = lx * res
        ly = ly * res
        L2 = line([lx[0],ly[0]], [lx[-1], ly[-1]])

        lint = list(intersection(L1, L2))

        opp = find_farthest(lx, ly, lint)

        dx, dy = find_translation(lint, opp)
        new_lx = np.asarray(plx + dx)
        new_ly = np.asarray(ply + dy)

        wm_coords['x'] = pia_coords['x']
        wm_coords['y'] = pia_coords['y']

        pia_coords['x'] = new_lx
        pia_coords['y'] = new_ly

        theta, offset = upright_angle(layer_coords, soma_coords, pia_coords, wm_coords)
        theta += np.pi
        soma_node = morph.compartment_list_by_type(1)[0]
        aff = [1., 0., 0., 0., 1., 0., 0., 0., 1., -soma_node["x"], -soma_node["y"], -soma_node["z"]]
        morph.apply_affine(aff)
        aff = [np.cos(theta), -np.sin(theta), 0., np.sin(theta), np.cos(theta), 0., 0., 0., 1., 0., -offset, 0.]
        morph.apply_affine(aff)

        print("\tsaving uprighted morph {}".format(specimen_id))
        morph.save(uout)

        flip = determine_mirror(lint, plx, ply, lx, ly)

        if flip:
            print("\tflipping morph {}".format(specimen_id))
            mdict = to_dict(uout)
            t = pd.DataFrame.from_dict(mdict).T
            t.x = t.x * -1
            tdict = t.to_dict(orient = 'index')
            tmorph = dict_to_Morphology(tdict)
            tmorph.save(uout)
    
        return error_dict
    
    except:
        print("ERROR: Unknown issue with cell {}", specimen_id)
        error_dict[specimen_id] = "Unknown issue with this cell"
        return error_dict


In [5]:
#upright all cells in dataframe
cells_with_issues = {}
for ix, row in df.iterrows():
    
    specimen_id = row[specimen_id_column]
    oout = os.path.join(o_out, "{}.swc".format(specimen_id))
    uout = os.path.join(upright_out, "{}_upright.swc".format(specimen_id))

    cells_with_issues= upright_nrn(specimen_id, oout, uout, cells_with_issues)


1102613885
	saving uprighted morph 1102613885
	flipping morph 1102613885
1102623321
	saving uprighted morph 1102623321
1110880399
	saving uprighted morph 1110880399
	flipping morph 1110880399
1110896667
	saving uprighted morph 1110896667
	flipping morph 1110896667
1112128140
	saving uprighted morph 1112128140
1112760246
	saving uprighted morph 1112760246
1112741472
	saving uprighted morph 1112741472
1096755367
	saving uprighted morph 1096755367
1096738774
	saving uprighted morph 1096738774
1112811000
	saving uprighted morph 1112811000
1096716775
	saving uprighted morph 1096716775
1096732440
	saving uprighted morph 1096732440
1079081411
ERROR: Couldn't load layers for 1079081411
1129691862
	saving uprighted morph 1129691862
	flipping morph 1129691862
1129687286
	saving uprighted morph 1129687286
	flipping morph 1129687286
1129673408
	saving uprighted morph 1129673408
	flipping morph 1129673408
1133704467
	saving uprighted morph 1133704467
	flipping morph 1133704467
1133695858
	saving up

In [6]:
cells_with_issues

{'1079081411': 'Could not load layers'}